# Looming Analysis — Example

Minimal usage of the `looming_analysis` package:

1. Load `.braidz` files grouped by experimental condition.
2. Classify trials as responsive / non-responsive.
3. Plot across the parameter space using the `row_by` / `col_by` / `hue_by` API.

In [ ]:
%matplotlib inline
%load_ext autoreload
%autoreload 2

import matplotlib.pyplot as plt
import numpy as np

from looming_analysis import (
    classify_responsiveness,
    plot_heading_changes,
    plot_responses,
    plot_responses_by_responsiveness,
    plot_responsiveness_rates,
    process_file_groups,
)

## Load data

`file_groups` maps each experimental condition to a list of `.braidz` paths. Every response returned gets a `'group'` key you can later use with `hue_by='group'`.

In [ ]:
root_folder = "/mnt/data/experiments"
file_groups = {
    "CS": [
        "/mnt/data/experiments/20260420_171221.braidz",
        "/mnt/data/experiments/20260421_145231.braidz",
        "/mnt/data/experiments/20260422_153355.braidz",
    ],
    "Empty-Split": [
        "/mnt/data/experiments/20260423_152419.braidz",
        "/mnt/data/experiments/20260424_151230.braidz",
        "/mnt/data/experiments/20260425_145046.braidz",
    ],
}

responses = process_file_groups(
    file_groups,
    pre_frames=-10,
    post_frames=50,
    debug=False,
    max_gap_frames=5,
    include_sham=True,
    cache_dir=".braidz_cache",
)

In [ ]:
classify_responsiveness(
    responses,
    heading_threshold_deg=45,
    threshold_deg_s=500,
    baseline_window_ms=(-90, -10),
)

## Comparing classification methods

In [ ]:
methods = [
    "is_responsive",
    "is_responsive_zscore",
    "is_responsive_heading",
    "is_responsive_saccade",
    "is_responsive_impulse",
    "is_responsive_combined",
]

for label, subset in [
    ("SHAM", [r for r in responses if r.get("sham")]),
    ("REAL", [r for r in responses if not r.get("sham")]),
]:
    print(f"\n{label} (n={len(subset)}):")
    for m in methods:
        n = sum(1 for r in subset if r.get(m))
        print(f"  {m:<35} {n:4d} / {len(subset)}  ({100 * n / len(subset):.1f}%)")

## Faceted angular velocity traces

`plot_responses` maps:
- `row_by` → subplot rows
- `col_by` → subplot columns
- `hue_by` → line colors within each subplot

If you have more stimulus parameters than you can fit, pre-filter the list: `[r for r in responses if r['expansion_duration_ms'] == 500]`.

In [ ]:
fig = plot_responses(
    responses,
    col_by="stimulus_offset_deg",
    hue_by="group",
    row_by="is_responsive",
    show_sham_baseline=True,
)

## Responsiveness rates

For bar / violin plots, `col_by` becomes the x-axis tick dimension and `hue_by` gives side-by-side bars/violins within each tick. Use `row_by` to facet by an additional parameter.

In [ ]:
fig = plot_responsiveness_rates(
    responses,
    col_by="stimulus_offset_deg",
    hue_by="group",
)

## Heading change distribution

Violin plot of the net heading change (degrees) per trial. Same `col_by` / `hue_by` semantics as the responsiveness rate plot above. `n` is shown above each violin.

In [ ]:
fig = plot_heading_changes(
    responses,
    col_by="stimulus_offset_deg",
    hue_by="group",
    absolute=True,
    responsive_only=True,
)

## Responsive vs non-responsive traces

`plot_responses_by_responsiveness` splits on the `is_responsive` field.
To use a different criterion, copy it into `is_responsive` before plotting
(and restore it after if you want to keep the original).

Rows are always Responsive / Non-responsive (no `row_by` parameter).
Columns and hue work as in `plot_responses`.

In [ ]:
# Default: split by peak |ω| criterion (is_responsive)
fig = plot_responses_by_responsiveness(
    responses,
    col_by="stimulus_offset_deg",
    hue_by="group",
)

# To split by a different criterion, use a shallow copy so the original
# is_responsive field is not overwritten:
criterion = "is_responsive_zscore"  # or _zscore / _heading / _impulse
responses_alt = [{**r, "is_responsive": r[criterion]} for r in responses]
fig = plot_responses_by_responsiveness(
    responses_alt, col_by="stimulus_offset_deg", hue_by="group"
)

## Saccade latency distribution

`saccade_onset_ms` (method 3) records when the saccade started relative to stimulus onset.
Plotting it relative to `end_expansion_time` shows whether flies tend to respond
before, during, or after the looming reaches maximum size.

In [ ]:
def find_turn(r, end_offset_s=0.2, threshold_deg_s=500.0):
    time = r["time"]
    ang_vel_abs = np.abs(np.rad2deg(r["ang_vel"]))
    end_t = r["end_expansion_time"]
    mask = (time >= 0) & (time <= end_t + end_offset_s)
    vals = ang_vel_abs[mask]
    if vals.size == 0 or np.all(np.isnan(vals)):
        return False, float("nan")
    peak_local = int(np.nanargmax(vals))
    peak_val = float(vals[peak_local])
    turn_time_ms = float(time[mask][peak_local] * 1000)
    return peak_val >= threshold_deg_s, turn_time_ms


for r in responses:
    r["turned"], r["turn_time_ms"] = find_turn(r)

fig, ax = plt.subplots(figsize=(7, 4))
for group in sorted({r["group"] for r in responses}):
    times = [
        r["turn_time_ms"] for r in responses if r["group"] == group and r["turned"]
    ]
    ax.hist(times, bins=30, alpha=0.6, label=f"{group} (n={len(times)})")

ax.axvline(300, color="k", linestyle="--", alpha=0.7, label="end expansion (300 ms)")
ax.set_xlabel("Turn time from stimulus onset (ms)")
ax.set_ylabel("Count")
ax.legend()
fig.tight_layout()